In [28]:
# Create environment using GPU:
# conda create --name gpu_offline_model python=3.10
# conda activate gpu_offline_model
# conda install ipykernel
# python -m ipykernel install --user --name=gpu_offline_model --display-name="GPU Model Offline"



# Steps for RAG based real time search engine (For deep search add extra functionality like add google research, wiki, other resarch article and local documents etch)
- Get user queries
- Transform to detailed prompt
- Identify to query type and provide option incase on multiple scenario : (single request or requeired multile sub query or required a road map to identify the accurate details. For example if we write a blog we need all necessary step which is essential to write a blog:
    - Writing a blog for website promotion and SEO requires a strategic approach that goes beyond just creating good content. The focus is on attracting traffic, engaging readers, and converting them into customers
    - Writing a blog for website promotion and SEO requires a strategic approach that goes beyond just creating good content. The focus is on attracting traffic, engaging readers, and converting them into customers   ---> this giving a different scenario
    - There are several key pointers to consider while writing a blog for a website.   ---> more focus on content writing
- Based on query selection generate AI result (result - A)
- Based on query selection define search type: Detailed, Moderate or Basic
- Start online search with various method (need to check all methods - web search, web scrapping etc, realtime results from web etc)
- For detailed/moderate one - generate summary and Store in vectorDB | Remove duplicate details | Rank the results
- Identify relative information from RAG
- Prepare answer from filtered RAG

In [ ]:
import os
from dotenv import load_dotenv
from langchain_google_genai import ChatGoogleGenerativeAI

# Load the environment variables from the .env file
load_dotenv()

# Retrieve the API keys from the environment
api_key_1 = os.getenv("GEMINI_API_KEY_1")
api_key_2 = os.getenv("GEMINI_API_KEY_2")

model1 = ChatGoogleGenerativeAI(model='gemini-2.5-flash-lite', google_api_key = api_key_1)
model2 = ChatGoogleGenerativeAI(model='gemini-2.5-flash-lite', google_api_key = api_key_2)

In [ ]:
from langchain.prompts import ChatPromptTemplate

# RAG-Fusion: Related
option_template = """You are a helpful assistant and your task is to understand user's query and if required generates multiple search queries based on a single input query. \n
Only if required, generate multiple search queries related to: {question} \n
if user query is understandabe so generate lesser Output (maximum 6 queries):
note: only provide queries in output, no other content"""
prompt_option = ChatPromptTemplate.from_template(option_template)

from langchain_core.output_parsers import StrOutputParser

generate_options = prompt_option | model1 | StrOutputParser() | (lambda x: x.split("\n")
)

question = input()
options = generate_options.invoke({'question':question})

options


In [26]:
# Offline model for text to text task
import torch
from transformers import pipeline
from langchain_huggingface import HuggingFacePipeline
from langchain_community.document_loaders import WebBaseLoader, SeleniumURLLoader
from selenium.common.exceptions import WebDriverException
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_core.prompts import PromptTemplate
from langchain.chains.summarize import load_summarize_chain

# Define the URL of the website to load
url = "https://python.langchain.com/docs/integrations/retrievers/cohere-reranker/#doing-reranking-with-coherererank"

# --- 1. Load the webpage content ---
docs = None
try:
    # Use WebBaseLoader for a fast, initial attempt on static content
    loader = WebBaseLoader(web_paths=[url])
    docs = loader.load()
except Exception:
    print("WebBaseLoader failed, trying SeleniumURLLoader...")
    try:
        loader = SeleniumURLLoader(urls=[url])
        docs = loader.load()
    except WebDriverException as e:
        print(f"Selenium failed. Is your browser driver correctly installed? Error: {e}")
        #return f"Selenium failed. Is your browser driver correctly installed? Error: {e}"


# --- 2. Split the document into chunks ---
# This is crucial for large web pages to fit within the model's context window.
print("Splitting document into chunks...")
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=400,
    chunk_overlap=50
)
split_docs = text_splitter.split_documents(docs)

# Combine the content into a single string for generation
# For very large content, consider a map-reduce summarization approach.
combined_text = " ".join([doc.page_content for doc in split_docs])


# Check for and set device to GPU if available
device = 0 if torch.cuda.is_available() else -1
print(f"Using device: {'cuda:0' if device == 0 else 'cpu'}")

# --- 3. Activating local saved model ---
print("Initializing Hugging Face text generation model on GPU...")
llm_pipeline = pipeline(
    "text2text-generation",
    model="google/flan-t5-large",
    max_length=500,  # Adjust based on your needs
    device=device     # Use the configured device (GPU or CPU)
)

# Wrap the Hugging Face pipeline in a LangChain LLM object
llm = HuggingFacePipeline(pipeline=llm_pipeline)

# --- 4. Define prompt templates for the Map-Reduce steps ---
map_template = """
You are an expert at creating concise technical summaries.
Summarize the following chunk of text for a technical documentation page:
Content:
{text}
CONCISE SUMMARY:
"""
map_prompt = PromptTemplate.from_template(map_template)

reduce_template = """
You are an AI assistant that is an expert at creating professional technical documentation.
Based on the following summaries, write a single, cohesive, and easy-to-read technical documentation page.

Summaries:
{text}

---

Professional Documentation:
"""
reduce_prompt = PromptTemplate.from_template(reduce_template)

# --- 5. Combine into a Map-Reduce summarization chain ---
print("Generating documentation using Map-Reduce chain...")
chain = load_summarize_chain(
    llm,
    chain_type="map_reduce",
    map_prompt=map_prompt,
    combine_prompt=reduce_prompt,
    verbose=True # Set to True to see the intermediate steps
)

# --- Generate the documentation ---
documentation = chain.invoke({"input_documents": split_docs})


Splitting document into chunks...


In [ ]:
import torch
from langchain_community.llms import Ollama
from langchain_core.prompts import PromptTemplate
from langchain.agents import initialize_agent, AgentType
from langchain.tools import DuckDuckGoSearchRun, Tool
from transformers import pipeline
from langchain_huggingface import HuggingFacePipeline

# # Check for and set device to GPU if available
device = 0 if torch.cuda.is_available() else -1
print(f"Using device: {'cuda:0' if device == 0 else 'cpu'}")

# --- Step 1: Initialize the open-source LLM via Ollama ---
print("Initializing Hugging Face text generation model on GPU...")
llm_pipeline = pipeline(
    "text2text-generation",
    model="google/flan-t5-large",
    max_length=500,  # Adjust based on your needs
    device=device     # Use the configured device (GPU or CPU)
)

# Wrap the Hugging Face pipeline in a LangChain LLM object
llm = HuggingFacePipeline(pipeline=llm_pipeline)

# --- Step 2: Define the open-source tool for real-time data ---
# We use DuckDuckGoSearchRun to perform a real-time web search.
search = DuckDuckGoSearchRun()

# Convert the search functionality into a LangChain tool.
search_tool = Tool(
    name="duckduckgo_search",
    func=search._run,
    description="Useful for when you need to answer questions about current events or up-to-date information."
)

In [7]:
tools = [search_tool]

# --- Step 3: Initialize the Agent ---
# The LLM will use the tool description to decide when to call it.
agent_executor = initialize_agent(
    tools,
    llm,
    agent=AgentType.ZERO_SHOT_REACT_DESCRIPTION,
    verbose=True,
    handle_parsing_errors=True # Add this parameter
)

# --- Step 4: Run the agent with a real-time query ---
user_query = input()
print("\n--- Running Agent with Real-Time Query ---")
response = agent_executor.run(user_query)
print(f"\nAI Response: {response}")


 what is the capital of india



--- Running Agent with Real-Time Query ---


> Entering new AgentExecutor chain...


Token indices sequence length is longer than the specified maximum sequence length for this model (728 > 512). Running this sequence through the model will result in indexing errors


['Ahmedabad', 'Ahmedabad', 'Ahmedabad', 'Ahmedabad', 'Ahmedabad', 'Ahmedabad', 'Ahmedabad', 'Ahmedabad', 'Ahmedabad', 'Ahmedabad', 'Ahmedabad', 'Ahmedabad', 'Ahmedabad', 'Ahmedabad', 'Ahmedabad', 'Ahmedabad', 'Ahmedabad', 'Ahmedabad', 'Ahmedabad', 'Ahmedabad', 'Ahmedabad', 'Ahmedabad', 'Ahmedabad', 'Ahmedabad', 'Ahmedabad', 'Ahmedabad', 'Ahmedabad', 'Ahmedabad', 'Ahmedabad', 'Ahmedabad', 'Ahmedabad', 'Ahmedabad', 'Ahmedabad', 'Ahmedabad', 'Ahmedabad', 'Ahmedabad', 'Ahmedabad', 'Ahmedabad', 'Ahmedabad', 'Ahmedabad', 'Ahmedabad', 'Ahmedabad', 'Ahmedabad', 'Ahmedabad', 'Ahmedabad', 'Ahmedabad', 'Ahmedabad', 'Ahmedabad', 'Ahmedabad', 'Ahmedabad', 'Ahmedabad', 'Ahmedabad', 'Ahmedabad', 'Ahmedabad', 'Ahmedabad', 'Ah
Observation: Invalid Format: Missing 'Action:' after 'Thought:
Thought:['Ahmedabad', 'Ahmedabad', 'Ahmedabad', 'Ahmedabad', 'Ahmedabad', 'Ahmedabad', 'Ahmedabad', 'Ahmedabad', 'Ahmedabad', 'Ahmedabad', 'Ahmedabad', 'Ahmedabad', 'Ahmedabad', 'Ahmedabad', 'Ahmedabad', 'Ahmedabad', 

KeyboardInterrupt: 

In [ ]:
# Offline version of embedding model
from langchain_huggingface import HuggingFaceEmbeddings
embeddings_model = HuggingFaceEmbeddings(model_name="intfloat/e5-base-v2")  # this is better compare to"all-MiniLM-L6-v2" and it use 768 dimensions
# embeddings_model = HuggingFaceEmbeddings(model_name="all-MiniLM-L6-v2")
# nomic-ai/nomic-embed-text-v1 -- This GPT-style model offers the highest top-5 retrieval accuracy and could be a good choice but its little heaver

# Doc embedding-----------------------------
documents = [
    "Delhi is the capital of India",
    "Kolkata is the capital of West Bengal",
    "Paris is the capital of France"
]
doc_embedding = embeddings_model.embed_documents(documents)
print(str(doc_embedding))